In [1]:

import os
os.environ['OMP_NUM_THREADS'] = '1'
os.environ['KMP_DUPLICATE_LIB_OK'] = 'TRUE'

import sys
import types
import pickle
import warnings
from pathlib import Path
import numpy as np
import pandas as pd
import librosa
from scipy.stats import skew, kurtosis
from scipy.spatial.distance import cdist
from sklearn.preprocessing import StandardScaler
from sklearn.decomposition import NMF


DATASET_DIR = Path('/kaggle/input/datasets/conatuszy/data2-2026/kaggle_upload')
MODELS_DIR  = DATASET_DIR / 'models'
COMP_DIR    = Path('/kaggle/input/competitions/birdclef-2026')
TEST_DIR    = COMP_DIR / 'test_soundscapes'


SR = 32000
CLIP_SEC = 5.0
N_SAMPLES = int(SR * CLIP_SEC)
N_FFT = 2048
HOP_LENGTH = 512
N_MELS = 128
FMIN = 20
FMAX = 16000
N_MFCC = 20



def load_full_audio(audio_path):
    y, _ = librosa.load(str(audio_path), sr=SR, mono=True)
    return y.astype(np.float32)

def extract_window_at_offset(y_full, offset_sec, win_samples=N_SAMPLES):
    start = int(offset_sec * SR)
    end = start + win_samples
    if end > len(y_full):
        start = max(0, len(y_full) - win_samples)
        end = start + win_samples
    y = y_full[start:end]
    if len(y) < win_samples:
        y = np.pad(y, (0, win_samples - len(y)))
    return y.astype(np.float32)


def compute_log_mel(y):
    mel = librosa.feature.melspectrogram(
        y=y, sr=SR, n_fft=N_FFT, hop_length=HOP_LENGTH,
        n_mels=N_MELS, fmin=FMIN, fmax=FMAX)
    return librosa.power_to_db(mel, ref=np.max)

def compute_stft_magnitude(y):
    return np.abs(librosa.stft(y, n_fft=N_FFT, hop_length=HOP_LENGTH))

def _stats4(matrix):
    return np.concatenate([
        matrix.mean(axis=1), matrix.std(axis=1),
        skew(matrix, axis=1), kurtosis(matrix, axis=1)])

def _segment_stats(matrix, n_segments=3):
    chunks = np.array_split(matrix, n_segments, axis=1)
    return np.concatenate([
        np.concatenate([c.mean(axis=1), c.std(axis=1)]) for c in chunks])

def extract_handcrafted_features(y):
    log_mel = compute_log_mel(y)
    stft_mag = compute_stft_magnitude(y)

    seg = _segment_stats(log_mel, 3)

    mfcc = librosa.feature.mfcc(S=log_mel, n_mfcc=N_MFCC)
    d1 = librosa.feature.delta(mfcc)
    d2 = librosa.feature.delta(mfcc, order=2)
    mfcc_f = np.concatenate([_stats4(mfcc), _stats4(d1), _stats4(d2)])

    chroma = librosa.feature.chroma_stft(S=stft_mag, sr=SR)
    chroma_f = np.concatenate([chroma.mean(1), chroma.std(1)])

    contrast = librosa.feature.spectral_contrast(S=stft_mag, sr=SR, n_bands=6)
    contrast_f = np.concatenate([contrast.mean(1), contrast.std(1)])

    ro = librosa.feature.spectral_rolloff(S=stft_mag, sr=SR)[0]
    ro_f = np.array([ro.mean(), ro.std(), skew(ro), kurtosis(ro)])

    bw = librosa.feature.spectral_bandwidth(S=stft_mag, sr=SR)[0]
    bw_f = np.array([bw.mean(), bw.std(), skew(bw), kurtosis(bw)])

    rms = librosa.feature.rms(y=y, frame_length=N_FFT, hop_length=HOP_LENGTH)
    rms_db = librosa.amplitude_to_db(rms, ref=np.max)[0]
    rms_f = np.array([rms_db.mean(), rms_db.std(), skew(rms_db), kurtosis(rms_db)])

    zcr = librosa.feature.zero_crossing_rate(y, frame_length=N_FFT, hop_length=HOP_LENGTH)[0]
    zcr_f = np.array([zcr.mean(), zcr.std(), skew(zcr), kurtosis(zcr)])

    rms2 = librosa.feature.rms(y=y, frame_length=N_FFT, hop_length=HOP_LENGTH)[0]
    rms2 = rms2 / (rms2.max() + 1e-8)
    centered = rms2 - rms2.mean()
    ac = np.correlate(centered, centered, mode='full')
    ac = ac[len(ac)//2:][:40]
    ac = ac / (ac[0] + 1e-8)
    mod = np.abs(np.fft.rfft(rms2, n=512))[:20]
    mod = mod / (mod.sum() + 1e-8)
    env_stats = np.array([rms2.mean(), rms2.std(), skew(rms2), kurtosis(rms2)])
    env_f = np.concatenate([env_stats, ac, mod]).astype(np.float32)

    feat = np.concatenate([seg, mfcc_f, chroma_f, contrast_f, ro_f, bw_f, rms_f, zcr_f, env_f])
    return np.nan_to_num(feat.astype(np.float32)), log_mel


# BoAW

class _Codebook:
    def __init__(self, centroids):
        self.centroids = centroids.astype(np.float32)
    def predict(self, X):
        n = X.shape[0]
        labels = np.empty(n, dtype=np.int32)
        batch = 4096
        for i in range(0, n, batch):
            chunk = X[i:i+batch]
            d2 = (chunk**2).sum(1, keepdims=True) - 2*chunk@self.centroids.T + (self.centroids**2).sum(1)
            labels[i:i+batch] = d2.argmin(1)
        return labels

class BaggingBoAW:
    def __init__(self, n_codebooks=3, k=256, random_state=42):
        self.n_codebooks = n_codebooks
        self.k = k
        self.random_state = random_state
        self.codebooks = []
    @property
    def feature_dim(self):
        return self.n_codebooks * self.k
    def transform(self, log_mel):
        frames = np.nan_to_num(log_mel.T.astype(np.float32))
        hists = []
        for cb, sc in self.codebooks:
            scaled = sc.transform(frames).astype(np.float32)
            labels = cb.predict(scaled)
            h, _ = np.histogram(labels, bins=self.k, range=(0, self.k))
            h = h.astype(np.float32) / (h.sum() + 1e-8)
            hists.append(h)
        return np.concatenate(hists)


# NMF

def _prepare_log_mel_for_nmf(log_mel):
    frames = log_mel.T.astype(np.float32)
    frames = frames - frames.min()
    frames = np.nan_to_num(frames)
    row_max = frames.max(axis=1, keepdims=True) + 1e-8
    frames = frames / row_max
    return np.clip(frames, 0, None)

class NMFFeatureExtractor:
    def __init__(self, n_components=64, random_state=42):
        self.n_components = n_components
        self.random_state = random_state
        self.nmf = None
    @property
    def feature_dim(self):
        return 3 * self.n_components
    def transform(self, log_mel):
        frames = _prepare_log_mel_for_nmf(log_mel)
        with warnings.catch_warnings():
            warnings.simplefilter("ignore")
            # on n'a besoin que de stats grossières (mean/std/max)
            # de H, donc peu d'itérations suffisent.
            try:
                self.nmf.max_iter = 30
            except Exception:
                pass
            H = self.nmf.transform(frames)
        feat = np.concatenate([H.mean(0), H.std(0), H.max(0)])
        return np.nan_to_num(feat.astype(np.float32))




_src = types.ModuleType('src'); sys.modules['src'] = _src
_boaw = types.ModuleType('src.boaw')
_boaw.BaggingBoAW = BaggingBoAW; _boaw._Codebook = _Codebook
sys.modules['src.boaw'] = _boaw; _src.boaw = _boaw
_nmf = types.ModuleType('src.nmf_features')
_nmf.NMFFeatureExtractor = NMFFeatureExtractor
_nmf._prepare_log_mel_for_nmf = _prepare_log_mel_for_nmf
sys.modules['src.nmf_features'] = _nmf; _src.nmf_features = _nmf



# Chargement des modèles

with warnings.catch_warnings():
    warnings.simplefilter("ignore")
    with open(MODELS_DIR / 'lgb_models.pkl', 'rb') as f:
        lgb_models = pickle.load(f)
    with open(MODELS_DIR / 'boaw_model.pkl', 'rb') as f:
        boaw_model = pickle.load(f)
    with open(MODELS_DIR / 'nmf_model.pkl', 'rb') as f:
        nmf_model = pickle.load(f)
    with open(MODELS_DIR / 'species_list.pkl', 'rb') as f:
        species_list = pickle.load(f)

active_idx = [i for i, sp in enumerate(species_list) if lgb_models.get(sp) is not None]
active_models = [lgb_models[species_list[i]] for i in active_idx]
print(f"Espèces : {len(species_list)} | LGB actifs : {len(active_idx)}")


# Inférence OPTIMISÉE

def extract_features_for_window(y_window):
    hc_feat, log_mel = extract_handcrafted_features(y_window)
    return np.concatenate([
        hc_feat, boaw_model.transform(log_mel), nmf_model.transform(log_mel)
    ]).astype(np.float32)

def predict_file(audio_path, segment_sec=5.0):
    y_full = load_full_audio(audio_path)
    duration = len(y_full) / SR
    n_segments = max(1, int(np.ceil(duration / segment_sec)))

    feats, end_secs = [], []
    for seg in range(n_segments):
        end_secs.append(int((seg + 1) * segment_sec))
        y_win = extract_window_at_offset(y_full, seg * segment_sec)
        feats.append(extract_features_for_window(y_win))
    X = np.nan_to_num(np.array(feats, dtype=np.float32))  # (n_segments, 2086)

    preds = np.zeros((len(end_secs), len(species_list)), dtype=np.float32)
    for k, model in zip(active_idx, active_models):
        preds[:, k] = model.predict(X, num_threads=4)

    return list(zip(end_secs, preds))



sample = pd.read_csv(COMP_DIR / 'sample_submission.csv')
test_files = sorted(TEST_DIR.glob("*.ogg"))
print(f"Fichiers de test : {len(test_files)}")

rows = []
for fi, audio_path in enumerate(test_files):
    file_id = audio_path.stem
    for end_sec, pred_vec in predict_file(audio_path):
        row = {'row_id': f"{file_id}_{end_sec}"}
        row.update({sp: pred_vec[i] for i, sp in enumerate(species_list)})
        rows.append(row)
    if (fi + 1) % 50 == 0:
        print(f"  {fi+1}/{len(test_files)}")

if len(rows) == 0:
    sample.to_csv('submission.csv', index=False)
    print("Aucun fichier test (mode interactif).")
else:
    sub_df = pd.DataFrame(rows)[sample.columns]
    sub_df.to_csv('submission.csv', index=False)
    print(f"submission.csv généré : {sub_df.shape}")

FileNotFoundError: [Errno 2] No such file or directory: '/kaggle/input/datasets/conatuszy/DATA2-2026/kaggle_upload/models/lgb_models.pkl'